# Pegasus-Large + LoRA Training and Generation
Train Pegasus-Large using LoRA adapters and generate meta-review predictions.

In [ ]:
!pip install -q transformers datasets accelerate peft sentencepiece

In [ ]:
import os, json, tempfile, torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
MODEL_NAME='google/pegasus-large'
TRAIN_FILE='/content/data/train.jsonl'
VAL_FILE='/content/data/val.jsonl'
TEST_FILE='/content/data/test.jsonl'
OUTPUT_DIR='pegasus_large_lora'

MAX_INPUT=768
MAX_TARGET=384

LORA_R=16
LORA_ALPHA=32
LORA_DROPOUT=0.1

os.makedirs(OUTPUT_DIR,exist_ok=True)

In [ ]:
def filter_jsonl_file(path):
    tmp=tempfile.NamedTemporaryFile(mode='w',delete=False,suffix='.jsonl')
    skipped=0
    with open(path,'r',encoding='utf-8') as f:
        for line in f:
            try:
                obj=json.loads(line)
                tmp.write(json.dumps(obj,ensure_ascii=False)+'\n')
            except:
                skipped+=1
    tmp.close()
    print(f'Skipped {skipped} lines')
    return tmp.name

TRAIN_FILE=filter_jsonl_file(TRAIN_FILE)
VAL_FILE=filter_jsonl_file(VAL_FILE)
TEST_FILE=filter_jsonl_file(TEST_FILE)

In [ ]:
dataset=load_dataset('json',data_files={'train':TRAIN_FILE,'validation':VAL_FILE})
print(dataset)

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
base_model=AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

base_model.gradient_checkpointing_enable()
base_model.config.use_cache=False

lora_config=LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj','v_proj'],
    bias='none'
)

model=get_peft_model(base_model,lora_config)
model.print_trainable_parameters()

device='cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

In [ ]:
def preprocess(example):
    inp=tokenizer(example['input_text'],max_length=MAX_INPUT,truncation=True,padding=False)
    lab=tokenizer(text_target=example['target_text'],max_length=MAX_TARGET,truncation=True,padding=False)
    labels=[t if t!=tokenizer.pad_token_id else -100 for t in lab['input_ids']]
    inp['labels']=labels
    return inp

tokenized=dataset.map(preprocess,remove_columns=dataset['train'].column_names)

collator=DataCollatorForSeq2Seq(tokenizer=tokenizer,model=model)

In [ ]:
training_args=TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    evaluation_strategy='steps',
    save_strategy='steps',
    eval_steps=300,
    save_steps=300,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    data_collator=collator
)

ckpt=get_last_checkpoint(OUTPUT_DIR)
if ckpt:
    trainer.train(resume_from_checkpoint=ckpt)
else:
    trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Training complete')

In [ ]:
from peft import PeftModel, PeftConfig
from tqdm import tqdm
import csv

peft_config=PeftConfig.from_pretrained(OUTPUT_DIR)

tokenizer=AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)
base=AutoModelForSeq2SeqLM.from_pretrained(peft_config.base_model_name_or_path)
model=PeftModel.from_pretrained(base,OUTPUT_DIR)
model=model.merge_and_unload().to(device)
model.eval()

def extract_decision(t):
    if 'DECISION:' in t:
        f=t.split('DECISION:')[1].split('\n')[0].upper()
        if f.startswith('ACCEPT'):return 'ACCEPT'
        if f.startswith('REJECT'):return 'REJECT'
    return 'UNKNOWN'

rows=[]
with open(TEST_FILE,'r') as f:
    for line in tqdm(f):
        ex=json.loads(line)
        inp=tokenizer(ex['input_text'],return_tensors='pt',truncation=True,max_length=768).to(device)
        with torch.no_grad():
            out=model.generate(**inp,max_new_tokens=300,num_beams=4,length_penalty=0.8)
        pred=tokenizer.decode(out[0],skip_special_tokens=True)
        rows.append({
            'paper_id':ex['paper_id'],
            'true_decision':extract_decision(ex['target_text']),
            'pred_decision':extract_decision(pred),
            'pred_meta_review':pred
        })

os.makedirs('data/predictions',exist_ok=True)
with open('data/predictions/pegasus_large_lora_predictions.csv','w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=rows[0].keys())
    w.writeheader()
    w.writerows(rows)
print('Predictions saved')